In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/

In [ ]:
from portrait_divergence import portrait_py

In [ ]:
cd /content/drive/MyDrive/gexf/NetLSD-master

In [ ]:
import netlsd

In [ ]:
import csv
import math
import os
import sys, os
import itertools
import numpy as np
import networkx as nx
from numpy.linalg import norm
import matplotlib.pyplot as plt
from numpy import linalg as LA
import pandas as pd
# from portrait_divergence import portrait_divergence

from statistics import mean


import json
import os
from scipy.spatial import distance
from scipy.stats import norm
from scipy.stats import entropy
from math import sqrt

In [ ]:
def AdjacencyMatrix(G):
    n = len(G.nodes())

    # Compute the adjacency matrix
    adjacency_matrix = nx.adjacency_matrix(G, weight='WEIGHT').todense()

    # Adj_two_dimensional_matrix = np.reshape(adjacency_matrix, (n, n))

    return adjacency_matrix

def Normalized_Laplac(G):
    n = len(G.nodes())
    # Compute the normalized Laplacian matrix
    normalized_laplacian_matrix = nx.normalized_laplacian_matrix(G).todense()

    # Lap_two_dimensional_matrix = np.reshape(normalized_laplacian_matrix, (n, n))
    return normalized_laplacian_matrix

In [ ]:
#-------------------- MONOLAYER ---------------

# column_names = ["nbOfNodes", "nbOfEdges", "density","Adjacency", "NormLaplac","B_scalar","fact"]
column_names = ["meanDegree", "betwenessCentrality", "density","Adjacency", "NormalizedLaplacien","B_scalar", "Cluster_Coefficient", "num_triads", "eccentricities", "max_closeness", "max_core", "NetLSD", "Spectra", "NEtMF"]


# compute network properties
def Properties(G, number_):
    #meanDegree = np.std(dict(G.degree()).values())
    meanDegree = np.std(list(dict(G.degree()).values())) # Convert dict_values to list
    #btwCentrality =  np.std(dict(nx.betweenness_centrality(G)).values())
    btwCentrality =  np.std(list(dict(nx.betweenness_centrality(G)).values())) # Convert dict_values to list
    density = nx.density(G)

    # adj = sum(np.linalg.eigvals(AdjacencyMatrix(G)))
    adj_eigenvectors = np.linalg.eigvals(AdjacencyMatrix(G))
    adj_eigenvectors = np.array([num.real if isinstance(num, complex) else num for num in adj_eigenvectors])
    # print(adj_eigenvectors)
    product_eigenvalues = np.std(adj_eigenvectors)

    # normLap = sum(np.linalg.eigvals(Normalized_Laplac(G)))
    Lap_eigenvectors = np.linalg.eigvals(Normalized_Laplac(G))
    Lap_eigenvectors = np.array([num.real if isinstance(num, complex) else num for num in Lap_eigenvectors])
    normLap = np.std(Lap_eigenvectors)

    portrait_B = portrait_py(G)
    # B_scalar = np.linalg.det(portrait_B)
    U, s, V = np.linalg.svd(portrait_B)
    B_scalar = np.std(s)
    # B_scalar = np.sum(portrait_B)

    clus_coefficient = nx.average_clustering(G)

    closness = nx.closeness_centrality(G)
    #max_closeness = np.std(closness.values())
    max_closeness = np.std(list(closness.values())) # Convert dict_values to list

    core = nx.core_number(G)
    max_core = np.std(list(core.values())) # Convert dict_values to list


    num_triads = nx.triangles(G)
    max_triads= np.std(list(num_triads.values())) #dentifying highly connected nodes, Convert dict_values to list

    eccentricities = nx.eccentricity(G)
    max_eccentricities= np.std(list(eccentricities)) # Convert dict_values to list

    NetLSD = netlsd.heat(G)
    # print(NetLSD)
    mean_NetLSD = np.std(NetLSD)

    spectra = nx.laplacian_spectrum(G)
    mean_spectra = np.std(spectra)

    netmf = np.load('/content/drive/MyDrive/gexf/output_NetMF/'+number_+'.npy')
    net =np.array(netmf).flatten()
    print(net)
    netmf_mean = np.std(net)

    return float(meanDegree), float(btwCentrality), float(density), float(product_eigenvalues), float(normLap), float(B_scalar), float(clus_coefficient), float(max_triads), float(max_eccentricities), float(max_closeness), float(max_core), float(mean_NetLSD), float(mean_spectra), float(netmf_mean)

def Property_Verctor(input, number_):
    df = pd.read_csv(input)
    Graphtype = nx.Graph()
    G = nx.from_pandas_edgelist(df, source = "SOURCE", target="TARGET",edge_attr='WEIGHT', create_using=Graphtype)

    nbOfNodes, nbOfEdges, density, product_eigenvalues, normLap , B_scalar, clus_coefficient, max_triads, max_eccentricities, max_closeness, max_core, mean_NetLSD, mean_spectra, netmf_mean = Properties(G, number_)
    vector = [nbOfNodes, nbOfEdges, density, product_eigenvalues, normLap, B_scalar, clus_coefficient, max_triads, max_eccentricities, max_closeness, max_core, mean_NetLSD, mean_spectra, netmf_mean]
    return vector



In [ ]:

#save features in 1 csv file "movie_features2.csv"

# Get the path to the 'gexf' folder
csv_folder_path = "//content/drive/MyDrive/gexf/csv"

# Create an empty dictionary to store the file names
csv_files = {}

# Iterate through the files in the 'gexf' folder
for filename in os.listdir(csv_folder_path):
    # Get the full path to the file
    file_path = os.path.join(csv_folder_path, filename)

    # Check if the file is a regular file (not a directory)
    if os.path.isfile(file_path):
        # Add the file name to the dictionary
        csv_files[filename] = file_path


# Define the header and initialize an empty list to store the rows
header = ['movie', 'meanDegree', 'betwenessCentrality', 'density', 'adj_eigenvector_centrality', 'NormalizedLaplacien', 'B_scalar', 'Cluster_Coefficient', 'number_of_Triads', 'max_eccentricities', 'max_closeness', 'max_core', 'NetLSD', 'mean_spectra', 'netmf']
rows = []


for i in csv_files.keys():
    parts = i.split('.')
    meanDegree, betwenessCentrality, density, product_eigenvalues, NormalizedLaplacien, B_scalar, clus_coefficient, max_triads, max_eccentricities, max_closeness, max_core, mean_NetLSD, mean_spectra, netmf_mean = Property_Verctor("/content/drive/MyDrive/gexf/csv/"+i, parts[0])
    # adj_eigenvectors = np.fromstring(adj_eigenvectors, sep=' ')
    row = [str(parts[0]), meanDegree, betwenessCentrality, density, product_eigenvalues, NormalizedLaplacien, B_scalar, clus_coefficient, max_triads, max_eccentricities, max_closeness, max_core, mean_NetLSD, mean_spectra, netmf_mean]
    rows.append(row)

# Create a DataFrame from the rows and header
df = pd.DataFrame(rows, columns=header)

# Write the DataFrame to a CSV file
df.to_csv('/content/drive/MyDrive/gexf/movie_features2.csv', index=False)
